# Aprendizaje supervisado y no supervisado con Big Data

InAlumnos  
Marcelo Ismael López Verdugo -------||     A00959089   

## Sesión spark y setup

### Librerías y sesión

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan, lit, desc, hour, dayofmonth, dayofweek
from pyspark.sql.functions import split, regexp_extract
from pyspark.sql.functions import collect_set
from pyspark.sql.types import DoubleType, FloatType
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import DecisionTreeClassifier  
from pyspark.ml.evaluation import MulticlassClassificationEvaluator



spark = SparkSession.builder \
    .appName("Analisis Ecommerce Octubre") \
    .config("spark.driver.memory", "6g") \
    .getOrCreate()

spark

In [2]:
import os
os.environ['JAVA_HOME'] 

'C:\\Program Files\\Eclipse Adoptium\\jdk-17.0.18.8-hotspot'

### Directorio del dataset

In [3]:
root=".."
file_path = root + "/datasets/2019-Oct.csv"

df = spark.read.option("header", True) \
               .option("inferSchema", True) \
               .csv(file_path)

df.show(5, truncate=False)

+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|event_time         |event_type|product_id|category_id        |category_code                      |brand   |price  |user_id  |user_session                        |
+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|2019-09-30 17:00:00|view      |44600062  |2103807459595387724|NULL                               |shiseido|35.79  |541312140|72d76fde-8bb3-4e00-8c23-a032dfed738c|
|2019-09-30 17:00:00|view      |3900821   |2053013552326770905|appliances.environment.water_heater|aqua    |33.2   |554748717|9333dfbd-b87a-4708-9857-6336556b0fcc|
|2019-09-30 17:00:01|view      |17200506  |2053013559792632471|furniture.living_room.sofa         |NULL    |543.1  |519107250|566511c2-e2e3-422b-b695-cf8e6e792ca8|
|2019-09-30 17:0

In [4]:
df.describe().show()

+-------+----------+--------------------+--------------------+-------------------+--------+-----------------+-------------------+--------------------+
|summary|event_type|          product_id|         category_id|      category_code|   brand|            price|            user_id|        user_session|
+-------+----------+--------------------+--------------------+-------------------+--------+-----------------+-------------------+--------------------+
|  count|  42448764|            42448764|            42448764|           28933155|36335756|         42448764|           42448764|            42448762|
|   mean|      NULL|1.0549932375842676E7|2.057404237936260...|               NULL|     NaN|290.3236606848809|5.335371475081686E8|                NULL|
| stddev|      NULL|1.1881906970608136E7|1.843926466140411...|               NULL|     NaN|358.2691553394021|1.852373817465431E7|                NULL|
|    min|      cart|             1000978| 2053013552226107603|    accessories.bag|  a-case|   

Eliminación de vacíos y nulos

In [5]:
print("=== Limpieza de valores vacíos y nulos ===")

# Reemplazar cadenas vacías por NULL solo en columnas string
string_cols = [c for c, t in df.dtypes if t == 'string']
df_clean = df.na.replace("", None, subset=string_cols) if string_cols else df

# Opcional: eliminar filas que tengan cualquier valor nulo
# Si quieres mantener filas con algunos valores faltantes, no uses dropna().
print(f"Filas antes de dropna: {df_clean.count()}")
df_clean = df_clean.dropna()
print(f"Filas después de dropna: {df_clean.count()}")

# Reutiliza df_clean en análisis posteriores si se desea
# df = df_clean

=== Limpieza de valores vacíos y nulos ===
Filas antes de dropna: 42448764
Filas después de dropna: 26560620


## 1. Teoría   
Las tareas de aprendizaje automático consisten en generar modelos matemáticos con la capacidad de predecir una variable o de encontrar un patrón.  En general los primeros suelen asociarse a un aprendizaje supervisado y los segundos a los no supervisados.

* Categórico.  Es posible que los predictores o las salidas sean categóricas (0 o 1 para referirse o no a la categoría).  Hay estrategias de codificación one-hot para colocar un 1 cuando se tiene la presencia de tal categoría y un 0 cuando sea el caso contrario.  Tal tratamiento sucede tambien en la salida donde se le da una ponderación de 0 a 1 de acuerdo a si pertenece o no a cierta categoría.  
* Numérico.  Por lo general son valores flotantes o numéricos que si bien pueden estar acotados, es usual que posean variables análogas.  Representarlas de forma categórica representaría una gran cantidad de valores únicos y su progresión posee un significado (un número más alto que otro significa algo, a diferencia de las categorías donde solo es hablar de otro tipo de grupo)

### Aprendizaje supervisado
El aprendizaje supervisado es aquel que, como su mismo nombre lo dice, requiere de supervisión.  La supervisión en éste caso consiste en darle conocimiento a priori de la salida esperada al modelo.  Dicha _supervisión_ es por lo general la presencia de un valor maestro, originalmente podría verse como una persona dando la etiqueta o el valor que se espera que el modelo prediga ante una entrada.  Tal entrada puede ser virtualmente cualquier cosa representable en modo numérico y la salida es el valor conocido a priori.  
El nacimiento del aprendizaje supervisado nace a partir de las regresiones donde se busca generalizar una salida a partir de entradas conocidas, pero con el tiempo se han diversificado los modelos, algoritmos y enfoques.  Por lo general hay algunas características a considerar en un aprendizaje supervisado:  


En general los modelos típicos para ello son:
* Regresión lineal.  Es cuando todas las variables son numéricas, se hace para predecir una variable numérica a partir de una variable numérica
* Regresión logística.  Es cuando se busca ver la pertenencia o no a una categoría.  Aqui es frecuente que las variables de entrada sean numéricas pero el modelo arroja una salida pseudo-binaria (donde 0 es la no pertenencia a la categoría y 1 es la pertenencia).
* Mixtos.  Puede presentarse un caso mixto donde una regresión lineal pueda portarse diferente dependiendo de la categoría a la que pertenece, no es usual pero es posible.


### Aprendizaje no supervisado
El aprendizaje no supervisado tiene la intención de encontrar información sin un conocimiento a priori.  Para este caso los algoritmos por lo general trabajan buscando algún tipo de agrupamiento ya que no existe una salida objetivo.  Lo que anteriormente se conocía como _supervisión_ ya no existe, de tal forma que el modelo puede encontrar patrones.  El aprendizaje no supervisado por lo general trabaja de forma muy parecida a la arquitectura de una regresión logística al tener variables de entrada que pueden ser flotantes mientras que su salida por lo comun es categórica.  Una aplicación usual es en etapas exploratorias de datos para encontrar patrones escondidos.  
Algunos ejemplos de modelos son:
* K-Means:  Consiste en colocar promedios móviles que minimicen la distancia de los puntos a su media más cercada.  Cada punto se agrupa a su respectiva media; el parámetro K corresponde a la cantidad de medias existentes y por lo general se hace uso de una cantidad de grupos que haga sentido y esté en coherencia con el punto de inflexión (o rodilla).  Tal punto de inflexión suele ser un punto donde agregar más medias móviles solo mejora sutilmente el error medio.
* Gaussian mixture model:  Muy parecido al modelo de K-means pero busca agrupar a nivel de probabilidades.  La salida de dicho modelo no es la agrupación per se, sino una probabilidad de que pertenezca o no a cierto grupo.  Muy útil para datos ambiguos con dos medias casi a la misma distancia.

### Implementación en pyspark.
Pyspark posee el módulo **ml** con paquetería de aprendizaje automático.  Es posible importar modelos y funciones muy parecidas a las que se haría en conjunto entre pandas y sklearn con la ventaja de operar a partir de particiones del dataframe.  Mezclado con estrategias de muestreo es posible ejecutar tareas de aprendizaje sobre bases de datos gigantes de forma más veloz.

## 2. Selección de datos (Muestreo)
Tras cargar la base de datos se realiza la partición de la misma.

In [6]:
df_part = df_clean

In [7]:
df_part = df_part.withColumn(
    "category_group",
    when(
        col("category_code").isNull(), 
        "unknown"
    ).otherwise(
        split(col("category_code"), "\\.").getItem(0)
    )
)

df_part.select("category_code", "category_group").show(10, truncate=False)

+-----------------------------------+--------------+
|category_code                      |category_group|
+-----------------------------------+--------------+
|appliances.environment.water_heater|appliances    |
|computers.notebook                 |computers     |
|electronics.smartphone             |electronics   |
|computers.desktop                  |computers     |
|apparel.shoes.keds                 |apparel       |
|electronics.smartphone             |electronics   |
|appliances.kitchen.microwave       |appliances    |
|electronics.smartphone             |electronics   |
|appliances.environment.water_heater|appliances    |
|furniture.bedroom.bed              |furniture     |
+-----------------------------------+--------------+
only showing top 10 rows


### Gestionando la cardinalidad
El dataset utilizado presenta columnas categóricas con una gran cantidad de valores diferentes.  Dicho comportamiento es comun en ventas y otros factores que puedan tener un enfoque de Pareto donde unos pocos grupos se pueden llevar gran cantidad de la participación, al mismo tiempo de tener otras categorías con poca participación.  
Para ejemplificarlo, es posible que existan hasta 40 marcas diferentes de celulares pero es posible que unas 8 marcas se lleven un 80 o 90% de la participación, por ende es posible generar una etiqueta de "Otros" para las categorías que no entren en el Top X.

In [8]:
# Agrupar valores de columnas no numéricas que no estén en el top 10.
# Las columnas con alta cardinalidad se transforman en <col>_grouped.
skip_columns = {"event_time", "category_group"}
non_numeric_columns = [name for name, dtype in df_part.dtypes if dtype == "string" and name not in skip_columns]

for col_name in non_numeric_columns:
    distinct_count = df_part.select(col_name).distinct().count()
    print(f"{col_name}: dtype=string, distinct={distinct_count}")

    top_values = [row[col_name] for row in df_part.groupBy(col_name)
                                              .count()
                                              .orderBy(desc("count"))
                                              .limit(10)
                                              .select(col_name)
                                              .collect()]

    grouped_col = f"{col_name}_grouped"
    df_part = df_part.withColumn(
        grouped_col,
        when(col(col_name).isin(top_values), col(col_name)).otherwise(lit("other"))
    )

print("Columnas agrupadas creadas:", [f"{c}_grouped" for c in non_numeric_columns])

event_type: dtype=string, distinct=3
category_code: dtype=string, distinct=126
brand: dtype=string, distinct=1731
user_session: dtype=string, distinct=6419693
Columnas agrupadas creadas: ['event_type_grouped', 'category_code_grouped', 'brand_grouped', 'user_session_grouped']


### Creando columnas auxiliares
Por comodidad futura se generan columnas auxiliares para usar datos de hora o día de la semana para predecir en los modelos.

In [12]:
# Creamos la columna 'dia_semana' (1 = Domingo, 7 = Sábado)
df_part = df_part.withColumn("week_day", dayofweek(col("event_time")))
# Crear columna weekhour combinando día de semana + hora/24
df_part = df_part.withColumn("weekhour", dayofweek(col("event_time")) + hour(col("event_time")) / 24)

### Creando muestra para 

In [14]:
#Dataset balanceado (muestreo por porcentaje de probabilidad de cada clase al 0.1%)
df_sample=df_part.sampleBy("event_type", fractions={"view": 0.001, "cart": 0.001, "purchase": 0.001}, seed=42)

### Comparativo entre dataset original y dataset del sample

In [13]:
#Dataset original
df_part.describe().show()

In [ ]:
#Dataset muestra M
df_sample.describe().show()

## 3. Preparación de datos

In [89]:
df_sample.describe().show()

+-------+----------+------------------+--------------------+-------------------+--------+------------------+--------------------+--------------------+--------------+------------------+------------------+------------------+---------------------+-------------+--------------------+
|summary|event_type|        product_id|         category_id|      category_code|   brand|             price|             user_id|        user_session|category_group|        dia_semana|          week_day|event_type_grouped|category_code_grouped|brand_grouped|user_session_grouped|
+-------+----------+------------------+--------------------+-------------------+--------+------------------+--------------------+--------------------+--------------+------------------+------------------+------------------+---------------------+-------------+--------------------+
|  count|     26654|             26654|               26654|              26654|   26654|             26654|               26654|               26654|         2

In [90]:
df_sample.show(10, truncate=False)

+-------------------+----------+----------+-------------------+-----------------------------+-------+------+---------+------------------------------------+--------------+----------+--------+------------------+-----------------------------+-------------+--------------------+
|event_time         |event_type|product_id|category_id        |category_code                |brand  |price |user_id  |user_session                        |category_group|dia_semana|week_day|event_type_grouped|category_code_grouped        |brand_grouped|user_session_grouped|
+-------------------+----------+----------+-------------------+-----------------------------+-------+------+---------+------------------------------------+--------------+----------+--------+------------------+-----------------------------+-------------+--------------------+
|2019-09-30 19:40:27|view      |1004133   |2053013555631882655|electronics.smartphone       |xiaomi |135.99|550308094|6ca60043-5b92-4827-ba8e-9759d47f6e68|electronics   |2    

In [95]:
columnas_categoricas = ["event_type","product_id","category_id", "brand","user_id", "category_group",'week_day','brand_grouped']

# Ejecuta una sola agregación dinámica
valores_unicos_full = df_part.agg(
    *(collect_set(col(c)).alias(c) for c in columnas_categoricas)
).collect()[0].asDict()

# Ver el resultado
for columna, valores in valores_unicos_full.items():
    print(f"Columna: {columna}, Valores únicos: {len(valores)}")

Columna: event_type, Valores únicos: 3
Columna: product_id, Valores únicos: 60371
Columna: category_id, Valores únicos: 248
Columna: brand, Valores únicos: 1731
Columna: user_id, Valores únicos: 2323036
Columna: category_group, Valores únicos: 13
Columna: week_day, Valores únicos: 7
Columna: brand_grouped, Valores únicos: 11


In [96]:
# Ejecuta una sola agregación dinámica
valores_unicos = df_sample.agg(
    *(collect_set(col(c)).alias(c) for c in columnas_categoricas)
).collect()[0].asDict()

# Ver el resultado
for columna, valores in valores_unicos.items():
    print(f"Columna: {columna}, Valores únicos: {len(valores)}")

Columna: event_type, Valores únicos: 3
Columna: product_id, Valores únicos: 7537
Columna: category_id, Valores únicos: 205
Columna: brand, Valores únicos: 799
Columna: user_id, Valores únicos: 25876
Columna: category_group, Valores únicos: 13
Columna: week_day, Valores únicos: 7
Columna: brand_grouped, Valores únicos: 11


## 4. Preparación de conjuntos entrenamiento, validación y prueba

In [97]:
# Dividir el conjunto muestreado en entrenamiento, validación y prueba.
# df_sample es un DataFrame de Spark, por lo que esta división se hace en Spark.
train_df, val_df, test_df = df_sample.randomSplit([0.7, 0.2, 0.1], seed=42)

print("Train count:", train_df.count())
print("Validation count:", val_df.count())
print("Test count:", test_df.count())

# En Spark ML, TrainValidationSplit se usa después con un estimador y un grid de parámetros.
# Para preparar modelos, sigue construyendo un Pipeline con VectorAssembler y StringIndexer.

Train count: 18612
Validation count: 5337
Test count: 2705


## 5. Aprendizaje automático

### 5.1 Aprendizaje supervisado   
Para el aprendizaje supervisado se hará uso de 

In [99]:
# ==========================================
# 1. DEFINICIÓN DE COLUMNAS
# ==========================================
# Identificamos cuáles son de texto (categóricas) y cuáles son numéricas
columnas_texto = ['brand_grouped']  # Tus variables de texto
columnas_numericas = ['week_day']  # Variables numéricas o ya procesadas

y_col = 'price'

# Asegurar que 'week_day' existe en el DataFrame
if "week_day" not in df_sample.columns:
    df_sample = df_sample.withColumn("week_day", dayofweek(col("columna_fecha")))

# First, find the maximum number of unique values in categorical columns
max_categories = 0
for col_name in columnas_texto:
    distinct_count = df_sample.select(col_name).distinct().count()
    print(f"{col_name}: {distinct_count} unique values")
    max_categories = max(max_categories, distinct_count)
print(f"\nMax categories found: {max_categories}")

# ==========================================
# 2. CONSTRUCCIÓN DE LAS ETAPAS DEL PIPELINE
# ==========================================
stages = []

# Etapa A: Indexar cada columna de texto de entrada de forma dinámica
columnas_indexadas = []
for col in columnas_texto:
    # Creamos un indexador para cada columna de texto (ej. 'columna1' -> 'columna1_indexed')
    indexer = StringIndexer(inputCol=col, outputCol=f"{col}_indexed", handleInvalid="skip")
    stages.append(indexer)
    # Guardamos el nombre de la nueva columna indexada para el VectorAssembler
    columnas_indexadas.append(f"{col}_indexed")

# Etapa B: Indexar la variable objetivo (y)
indexer_target = StringIndexer(inputCol=y_col, outputCol="label", handleInvalid="skip")
stages.append(indexer_target)

# Etapa C: Ensamblar todas las características
# Unimos las columnas de texto indexadas con las columnas numéricas
columnas_finales_entrada = columnas_indexadas + columnas_numericas

assembler = VectorAssembler(inputCols=columnas_finales_entrada, outputCol="features", handleInvalid="skip")
stages.append(assembler)

dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=5, maxBins=max_categories)
stages.append(dt)

# ==========================================
# 3. ENTRENAMIENTO Y EVALUACIÓN
# ==========================================
pipeline = Pipeline(stages=stages)

# División train/test
train_df, test_df = df_sample.randomSplit([0.8, 0.2], seed=42)

# Entrenar Pipeline
print("Entrenando el modelo...")
model_pipeline = pipeline.fit(train_df)
print("¡Modelo entrenado!")

# Predicciones
predictions = model_pipeline.transform(test_df)

# Evaluar Accuracy
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f" Precisión del modelo (Accuracy): {accuracy:.4f}")


brand_grouped: 11 unique values

Max categories found: 11
Entrenando el modelo...
¡Modelo entrenado!
 Precisión del modelo (Accuracy): 0.0185
